# Marketing table for customer analysis

- Add a RFM score to the table
- Create the final table
- Insert data into the final table

## Working query to aggregate the data

In [0]:
%skip
CREATE OR REPLACE TABLE data_lakehouse_project.platinium.customer_360 AS
WITH max_date AS (
  SELECT
    MAX(order_date) AS max_date
  FROM data_lakehouse_project.gold.fact_sales
),

base AS (
  SELECT
    c.customer_id,
    CONCAT(c.firstname, ' ', c.lastname) AS fullname,
    c.gender,
    c.birthdate,
    DATE_DIFF(year,c.birthdate, ANY_VALUE(m.max_date)) AS age,
    c.country,
    MIN(s.order_date) AS first_order,
    MAX(s.order_date) AS last_order,
    COUNT(DISTINCT s.order_number) AS total_orders,
    SUM(s.sales) AS total_sales,
    SUM(s.quantity) AS total_quantity,
    ROUND(SUM(s.sales) / COUNT(DISTINCT s.order_number), 2) AS avg_order_value,
    AVG(DATE_DIFF(s.due_date, s.order_date)) AS avg_days_between_delivery_and_order
  FROM data_lakehouse_project.gold.dim_customers AS c
  LEFT JOIN data_lakehouse_project.gold.fact_sales AS s
    ON c.customer_id = s.customer_id
  LEFT JOIN data_lakehouse_project.gold.dim_products AS p
    ON s.product_key = p.product_key
  JOIN max_date AS m
    ON TRUE
  GROUP BY ALL
),

top_category AS (
  SELECT customer_id, category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_subcategory AS (
  SELECT customer_id, sub_category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.sub_category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_product AS (
  SELECT customer_id, product_name, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.product_name,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

first_product AS (
  SELECT customer_id, product_name
  FROM (
    SELECT
      s.customer_id,
      p.product_name,
      s.order_date,
      ROW_NUMBER() OVER(PARTITION BY s.customer_id ORDER BY s.order_date ASC, s.order_number ASC) AS rnk
    FROM data_lakehouse_project.gold.fact_sales AS s
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p
      ON s.product_key = p.product_key
  )
  WHERE rnk = 1
),

rfm_scores AS (
  SELECT
    b.*,
    fp.product_name AS first_product_ordered,
    tc.category AS top_category,
    tsc.sub_category AS top_subcategory,
    tp.product_name AS top_product,
    DATEDIFF(MAX(b.last_order) OVER(), b.last_order) AS days_since_last_order,
    NTILE(5) OVER(ORDER BY b.last_order ASC) AS r_score,
    NTILE(5) OVER(ORDER BY b.total_orders ASC) AS f_score,
    NTILE(5) OVER(ORDER BY b.total_sales ASC) AS m_score,
    NTILE(5) OVER(ORDER BY b.avg_order_value ASC) AS v_score
  FROM base AS b
  LEFT JOIN top_category AS tc 
    ON b.customer_id = tc.customer_id
  LEFT JOIN top_subcategory AS tsc 
    ON b.customer_id = tsc.customer_id
  LEFT JOIN top_product AS tp 
    ON b.customer_id = tp.customer_id
  LEFT JOIN first_product AS fp
    ON b.customer_id = fp.customer_id
)

SELECT  
  rfm.*,
  (r_score + f_score + m_score + v_score) / 4 AS rfmv_score,
  CASE
    WHEN rfmv_score >= 4.5 THEN 'VIP'
    WHEN rfmv_score >= 3.5 THEN 'Premium'
    WHEN rfmv_score >= 2.5 THEN 'Standard'
    WHEN rfmv_score >= 1.5 THEN 'At Risk'
    ELSE 'Dormant'
    END AS rfmv_segment
FROM rfm_scores AS rfm
ORDER BY rfmv_score 

## Create the table & Insert data

In [0]:
CREATE OR REPLACE TABLE data_lakehouse_project.platinium.customer_360 AS
WITH max_date AS (
  SELECT
    MAX(order_date) AS max_date
  FROM data_lakehouse_project.gold.fact_sales
),

base AS (
  SELECT
    c.customer_id,
    CONCAT(c.firstname, ' ', c.lastname) AS fullname,
    c.gender,
    c.birthdate,
    DATE_DIFF(year,c.birthdate, ANY_VALUE(m.max_date)) AS age,
    c.country,
    MIN(s.order_date) AS first_order,
    MAX(s.order_date) AS last_order,
    COUNT(DISTINCT s.order_number) AS total_orders,
    SUM(s.sales) AS total_sales,
    SUM(s.quantity) AS total_quantity,
    ROUND(SUM(s.sales) / COUNT(DISTINCT s.order_number), 2) AS avg_order_value,
    AVG(DATE_DIFF(s.due_date, s.order_date)) AS avg_days_between_delivery_and_order
  FROM data_lakehouse_project.gold.dim_customers AS c
  LEFT JOIN data_lakehouse_project.gold.fact_sales AS s
    ON c.customer_id = s.customer_id
  LEFT JOIN data_lakehouse_project.gold.dim_products AS p
    ON s.product_key = p.product_key
  JOIN max_date AS m
    ON TRUE
  GROUP BY ALL
),

top_category AS (
  SELECT customer_id, category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_subcategory AS (
  SELECT customer_id, sub_category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.sub_category,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_product AS (
  SELECT customer_id, product_name, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.product_name,
      SUM(s.quantity) AS total_qty,
      ROW_NUMBER() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

first_product AS (
  SELECT customer_id, product_name
  FROM (
    SELECT
      s.customer_id,
      p.product_name,
      s.order_date,
      ROW_NUMBER() OVER(PARTITION BY s.customer_id ORDER BY s.order_date ASC, s.order_number ASC) AS rnk
    FROM data_lakehouse_project.gold.fact_sales AS s
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p
      ON s.product_key = p.product_key
  )
  WHERE rnk = 1
),

rfm_scores AS (
  SELECT
    b.*,
    fp.product_name AS first_product_ordered,
    tc.category AS top_category,
    tsc.sub_category AS top_subcategory,
    tp.product_name AS top_product,
    DATEDIFF(MAX(b.last_order) OVER(), b.last_order) AS days_since_last_order,
    NTILE(5) OVER(ORDER BY b.last_order ASC) AS r_score,
    NTILE(5) OVER(ORDER BY b.total_orders ASC) AS f_score,
    NTILE(5) OVER(ORDER BY b.total_sales ASC) AS m_score,
    NTILE(5) OVER(ORDER BY b.avg_order_value ASC) AS v_score
  FROM base AS b
  LEFT JOIN top_category AS tc 
    ON b.customer_id = tc.customer_id
  LEFT JOIN top_subcategory AS tsc 
    ON b.customer_id = tsc.customer_id
  LEFT JOIN top_product AS tp 
    ON b.customer_id = tp.customer_id
  LEFT JOIN first_product AS fp
    ON b.customer_id = fp.customer_id
)

SELECT  
  rfm.*,
  (r_score + f_score + m_score + v_score) / 4 AS rfmv_score,
  CASE
    WHEN rfmv_score >= 4.5 THEN 'VIP'
    WHEN rfmv_score >= 3.5 THEN 'Premium'
    WHEN rfmv_score >= 2.5 THEN 'Standard'
    WHEN rfmv_score >= 1.5 THEN 'At Risk'
    ELSE 'Dormant'
    END AS rfmv_segment
FROM rfm_scores AS rfm
ORDER BY rfmv_score 

In [0]:
%skip
SELECT 
  *
FROM data_lakehouse_project.platinium.customer_360

